# Lab: Why Language Is Hard, in Data

**Bluevision-ai Academy**
*Why Language Is Hard*

In lecture we named four hard problems, ambiguity, context dependency, scale and sparsity, and world knowledge, and watched four eras of NLP research break against them in turn. This lab puts three of those problems in your own hands as real data, then has you build a small system and spend fifteen minutes breaking it on purpose.

Each part runs the same rhythm the lecture did, just in cells instead of clicks: **Problem** (what we have, what we want, why the obvious tool doesn't reach it) -> **Ingredients** (the data, loaded, nothing else yet) -> **Idea** (the reasoning move, in words, before any code) -> **Build** (the code that embodies exactly that idea) -> **Payoff** (what just happened, and what it means). One idea per cell.

| Part | What you build | Ties back to |
|---|---|---|
| 1 -- Zipf's Law, For Real | Count a real corpus yourself; watch the power law, and its never-emptying tail, appear in your own numbers | Slides 13-14 |
| 2 -- Bag-of-Words Meets Meaning | Score sentence-pair similarity with plain word counts; watch synonyms score near zero and reversed sentences score perfect | Slide 24 |
| 3 -- One Word, Two Worlds | Pull real "bank" sentences out of the corpus; test whether nearby words alone can separate the two senses | Slides 8, 12 |
| Challenge | Build a keyword Q&A system over eight facts from this session, then break it on purpose | Slide 25 |
| Reflection | Five questions, answered in your own words | -- |

**Dataset:** real English prose from the NLTK Gutenberg corpus, eighteen public-domain books (Austen, Melville, Carroll, the King James Bible, and more), about 2.1 million words once you strip punctuation. Real text, not a toy example built to make a point. **Estimated time:** ~2 hours. Runs top to bottom on Google Colab with no setup beyond the cell below.

## Setup

Colab does not ship with NLTK's text corpora by default, so the cell below installs the libraries this lab needs and downloads the one corpus it uses throughout: `gutenberg`. It also loads the deck's own palette, so the plots here read like the ones you just watched.

In [1]:
%pip install -q --upgrade nltk scikit-learn plotly ipywidgets

import re
from collections import Counter

import numpy as np
import nltk
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from ipywidgets import Textarea, Button, VBox, Output
from IPython.display import display

nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not running on Colab -- local Jupyter with ipywidgets already works

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, DIM, GREEN = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#7f8c9a", "#1a7a4a"

print("Gutenberg fileids available:", gutenberg.fileids())

Note: you may need to restart the kernel to use updated packages.


Gutenberg fileids available: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']


## Part 1 -- Zipf's Law, For Real

**Problem.** Slide 13 put two numbers side by side: a ten-word sentence has about $10^{50}$ possible forms, but the largest training corpora hold only around $10^{13}$ sentences. A model cannot memorise its way through that gap; it has to generalise. Slide 14 then showed a log-log word-frequency plot to argue language is skewed in a very specific way, but that plot came from a corpus you never got to see. Is that skew a curated example, or does it show up the moment you count real text yourself?

**Ingredients.** All eighteen books in NLTK's Gutenberg corpus, concatenated into one list of tokens. We lowercase everything and keep only alphabetic tokens, dropping numbers and punctuation so a word and its capitalised twin count as one word.

In [2]:
raw_words = []
for fid in gutenberg.fileids():
    raw_words.extend(gutenberg.words(fid))

tokens = [w.lower() for w in raw_words if re.fullmatch(r"[a-zA-Z]+", w)]
vocab_counts = Counter(tokens)

print(f"corpus size: {len(tokens):,} tokens")
print(f"vocabulary size: {len(vocab_counts):,} distinct words")
print("ten most common words:", vocab_counts.most_common(10))

corpus size: 2,135,393 tokens
vocabulary size: 41,481 distinct words
ten most common words: [('the', 133583), ('and', 95442), ('of', 71267), ('to', 48057), ('a', 33960), ('in', 33580), ('i', 30265), ('that', 28798), ('he', 25857), ('it', 22303)]


**Idea -- rank them and look on a log-log scale.** Sort every word by how often it occurs, most common first. Zipf's Law claims frequency is roughly inversely proportional to rank: the second most common word occurs about half as often as the first, the third about a third as often, and so on. On ordinary axes that curve is an uninformative near-vertical drop. On log-log axes, an inverse-proportion relationship becomes a straight line, so plot the real data against an idealised $1/\text{rank}$ reference line and see how closely they track.

In [3]:
ranked = vocab_counts.most_common()
freqs = np.array([c for _, c in ranked], dtype=float)
ranks = np.arange(1, len(freqs) + 1)
ideal = freqs[0] / ranks  # the idealised 1/rank law, anchored at the top word's count

fig = go.Figure()
fig.add_trace(go.Scatter(x=ranks, y=freqs, mode="lines", name="observed",
                          line=dict(color=GOLD, width=2)))
fig.add_trace(go.Scatter(x=ranks, y=ideal, mode="lines", name="ideal 1/rank",
                          line=dict(color=BLUE, width=2, dash="dash")))
fig.update_layout(title="Word frequency vs. rank, 41,481 distinct words, log-log axes",
                   xaxis_title="rank", yaxis_title="frequency",
                   xaxis_type="log", yaxis_type="log", width=700, height=450)
fig.show()

# Fit the log-log slope over the first 5,000 ranks (the tail gets noisy one-off words)
N = 5000
slope, intercept = np.polyfit(np.log10(ranks[:N]), np.log10(freqs[:N]), 1)
print(f"fitted log-log slope over the first {N} ranks: {slope:.3f}  (Zipf's ideal is -1.0)")

hapax = sum(1 for c in vocab_counts.values() if c == 1)
print(f"hapax legomena (words seen exactly once): {hapax:,} "
      f"= {hapax / len(vocab_counts):.1%} of the vocabulary, "
      f"but only {hapax / len(tokens):.2%} of all tokens")

fitted log-log slope over the first 5000 ranks: -1.158  (Zipf's ideal is -1.0)
hapax legomena (words seen exactly once): 14,997 = 36.2% of the vocabulary, but only 0.70% of all tokens


**Payoff.** The observed curve hugs the idealised $1/\text{rank}$ line closely enough that the fitted slope, -1.16 over the first five thousand ranks, lands right next to Zipf's predicted -1.0. This was never a curated example: eighteen books by nine different authors, spanning three centuries and genres from scripture to whaling fiction to nonsense verse, all obey the same law. And the tail is exactly as extreme as slide 14 warned: 36% of the entire vocabulary, 14,997 distinct words, occurs exactly once, even though those one-off words make up less than 1% of the two-million-plus tokens actually read. The takeaway generalises well past this corpus: in any large body of natural-language text, a handful of function words carry most of the volume, and the majority of the vocabulary is desperately rare, which is precisely why a model that only counts frequencies runs out of data before it runs out of words.

**Idea -- does more data make the rare-word problem go away?** Slide 14's narration raised this directly: surely a bigger corpus would let the model eventually see every word enough times? Test it by growing the corpus in twenty equal slices, and after each slice, count how many distinct words have appeared *so far, and not before*. If the sparsity problem is fixable by scale, that count of brand-new words should fall toward zero as the corpus grows.

In [4]:
n_chunks = 20
chunk_size = len(tokens) // n_chunks
seen = set()
growth_x, growth_y = [], []
for i in range(n_chunks):
    chunk = tokens[i * chunk_size:(i + 1) * chunk_size]
    seen.update(chunk)
    growth_x.append((i + 1) * chunk_size)
    growth_y.append(len(seen))

fig = go.Figure(go.Scatter(x=growth_x, y=growth_y, mode="lines+markers", line=dict(color=TEAL)))
fig.update_layout(title="Distinct words seen so far, as the corpus grows",
                   xaxis_title="tokens read", yaxis_title="distinct words seen",
                   width=700, height=400)
fig.show()

first_new = growth_y[0]
last_new = growth_y[-1] - growth_y[-2]
print(f"first {chunk_size:,} tokens introduced {first_new:,} new words "
      f"(about 1 new word every {chunk_size / first_new:.0f} tokens)")
print(f"the LAST {chunk_size:,} tokens still introduced {last_new:,} new words "
      f"(about 1 new word every {chunk_size / last_new:.0f} tokens)")

first 106,769 tokens introduced 5,887 new words (about 1 new word every 18 tokens)
the LAST 106,769 tokens still introduced 2,233 new words (about 1 new word every 48 tokens)


**Payoff.** The curve never flattens to horizontal. The first hundred-thousand-odd tokens introduce a new word roughly every 18 tokens; by the final slice, two million tokens in, the corpus is still coughing up a new word every 48 tokens or so. That is real slowdown, but it is not convergence: at no point does the line go flat. This is slide 14's warning made numeric: growing the corpus does not empty the rare-word tail, it just stretches it thinner. The generalisable lesson is not "collect more data and sparsity disappears"; it is that any language model, no matter how large its training corpus, will keep meeting words and combinations it has never seen, which is exactly why slide 13 called generalisation the whole problem rather than a nice-to-have.

## Part 2 -- Bag-of-Words Meets Meaning

**Problem.** Slide 24 left us with a specific, damning claim: one-hot vectors give "cat" and "feline" a similarity of exactly zero, because two different words can never share a dimension. A bag-of-words vector is one step up from one-hot, it counts real sentences instead of single words, so maybe it recovers some of what one-hot loses. Test that directly: build sentence pairs that mean the same thing in different words, and sentence pairs that share words but mean different things, then see whether bag-of-words similarity tracks meaning or just tracks vocabulary overlap.

**Ingredients.** Ten hand-built sentence pairs. Five are paraphrases, the same event described with different words. Five share vocabulary but describe different or even reversed situations, including the deck's own polysemous words ("bank", "set") and a straightforward subject/object swap.

In [5]:
pairs = [
    ("The cat sat on the mat.", "The feline rested on the rug.", "same meaning"),
    ("He bought a new phone.", "He purchased a new mobile.", "same meaning"),
    ("The movie was boring.", "The film was dull.", "same meaning"),
    ("She is happy today.", "She feels joyful today.", "same meaning"),
    ("The car crashed into the wall.", "The automobile collided with the wall.", "same meaning"),
    ("The dog chased the cat up the tree.", "The cat chased the dog up the tree.", "different meaning"),
    ("The doctor examined the nurse.", "The nurse examined the doctor.", "different meaning"),
    ("The bank raised interest rates.", "The bank of the river flooded.", "different meaning"),
    ("Set the table for dinner.", "Set the record straight.", "different meaning"),
    ("I saw the man with the telescope.", "I bought a telescope with my own money.", "different meaning"),
]
for a, b, label in pairs:
    print(f"[{label:16s}]  {a!r}  <->  {b!r}")

[same meaning    ]  'The cat sat on the mat.'  <->  'The feline rested on the rug.'
[same meaning    ]  'He bought a new phone.'  <->  'He purchased a new mobile.'
[same meaning    ]  'The movie was boring.'  <->  'The film was dull.'
[same meaning    ]  'She is happy today.'  <->  'She feels joyful today.'
[same meaning    ]  'The car crashed into the wall.'  <->  'The automobile collided with the wall.'
[different meaning]  'The dog chased the cat up the tree.'  <->  'The cat chased the dog up the tree.'
[different meaning]  'The doctor examined the nurse.'  <->  'The nurse examined the doctor.'
[different meaning]  'The bank raised interest rates.'  <->  'The bank of the river flooded.'
[different meaning]  'Set the table for dinner.'  <->  'Set the record straight.'
[different meaning]  'I saw the man with the telescope.'  <->  'I bought a telescope with my own money.'


**Idea -- represent each sentence as a bag of words, then measure cosine similarity.** `CountVectorizer` turns every sentence into a vector of word counts over the shared vocabulary of both sentences, dropping stopwords ("the", "a", "is") so the comparison rests on content words alone. Cosine similarity then measures how much two count vectors point in the same direction: 1.0 for identical bags of words, 0.0 for no shared words at all. If bag-of-words tracked meaning, the five "same meaning" pairs should score consistently higher than the five "different meaning" pairs.

In [6]:
vec = CountVectorizer(stop_words="english")
vec.fit([s for pair in pairs for s in (pair[0], pair[1])])

results = []
for a, b, label in pairs:
    X = vec.transform([a, b])
    sim = cosine_similarity(X[0], X[1])[0, 0]
    results.append((sim, label, a, b))
    print(f"{sim:.3f}  [{label:16s}]  {a!r}  <->  {b!r}")

same_scores = [s for s, l, _, _ in results if l == "same meaning"]
diff_scores = [s for s, l, _, _ in results if l == "different meaning"]
print(f"\nmean similarity, same-meaning pairs:      {np.mean(same_scores):.3f}")
print(f"mean similarity, different-meaning pairs: {np.mean(diff_scores):.3f}")

0.000  [same meaning    ]  'The cat sat on the mat.'  <->  'The feline rested on the rug.'
0.333  [same meaning    ]  'He bought a new phone.'  <->  'He purchased a new mobile.'
0.000  [same meaning    ]  'The movie was boring.'  <->  'The film was dull.'
0.408  [same meaning    ]  'She is happy today.'  <->  'She feels joyful today.'
0.333  [same meaning    ]  'The car crashed into the wall.'  <->  'The automobile collided with the wall.'
1.000  [different meaning]  'The dog chased the cat up the tree.'  <->  'The cat chased the dog up the tree.'
1.000  [different meaning]  'The doctor examined the nurse.'  <->  'The nurse examined the doctor.'


0.333  [different meaning]  'The bank raised interest rates.'  <->  'The bank of the river flooded.'
0.333  [different meaning]  'Set the table for dinner.'  <->  'Set the record straight.'
0.333  [different meaning]  'I saw the man with the telescope.'  <->  'I bought a telescope with my own money.'

mean similarity, same-meaning pairs:      0.215
mean similarity, different-meaning pairs: 0.600


**Payoff.** The averages run backwards from what a working similarity measure should show: the five genuine paraphrases average 0.215, while the five different-meaning pairs average 0.600, nearly three times higher. Two paraphrases, cat/feline and movie/film, score exactly 0.0 despite meaning the same thing, the "cat" and "feline" villain from slide 24 reappearing at sentence scale. And two different-meaning pairs, the reversed dog/cat and doctor/nurse sentences, score a perfect 1.0, because a bag of words has no notion of word order or grammatical role: "the dog chased the cat" and "the cat chased the dog" are, to a count vector, the identical bag. The takeaway is not narrowly about synonyms: any representation built purely from which words are present, with no memory of order, structure, or the relationships between words that Firth's slide 12 quote pointed to, will systematically confuse "shares vocabulary" with "means the same thing", in both directions.

## Part 3 -- One Word, Two Worlds

**Problem.** Slide 8 built its whole argument for lexical ambiguity on one sentence: "I went to the bank." A single invented example proves the *possibility* of ambiguity, but slide 8 also claimed this is "not a rare edge case", that it is structural, pervasive, happening constantly in ordinary prose. Pull every real sentence containing "bank" out of the Gutenberg corpus and see for yourself whether both senses genuinely show up, unprompted, across genuinely different books.

**Ingredients.** Sixteen real sentences pulled from the corpus, each one containing the word "bank" or "banks", drawn from eight different books. We hand-label each one by its sense, riverbank/embankment, or financial institution, using nothing but ordinary reading comprehension; nothing here was written to make a point, every sentence is quoted exactly as it appears in its source book.

In [7]:
bank_sentences = [
    ("austen-persuasion", "water", "Anne found a nice seat for her, on a dry sunny bank, "
     "under the hedge-row, in which she had no doubt of their still being, in some spot or other."),
    ("bryant-stories", "water", "The minute the fox reached the bank he threw back his head, and gave a snap!"),
    ("bryant-stories", "water", "Then he ran swiftly along the river bank, to hunt for crabs; "
     "the Camel began to eat sugar-cane."),
    ("burgess-busterbrown", "water", "Little Joe sat down on the bank and prepared to enjoy his breakfast."),
    ("burgess-busterbrown", "water", "He was just climbing up the bank with the fat trout in his mouth."),
    ("carroll-alice", "water", "They were indeed a queer-looking party that assembled on the bank, "
     "the birds with draggled feathers, the animals with their fur clinging close to them, "
     "and all dripping wet, cross, and uncomfortable."),
    ("chesterton-ball", "water", "He suddenly swung himself up the high bank on one side of the lane."),
    ("edgeworth-parents", "water", "Susan sat down upon the bank in silent sorrow; her little brothers "
     "ran up to the butcher, and demanded whether he was going to do any harm to the lamb."),
    ("melville-moby_dick", "money", "There goes three thousand dollars, men! A bank! A whole bank!"),
    ("chesterton-thursday", "money", "An omnibus going to the Bank went rattling by with an unusual rapidity."),
    ("chesterton-thursday", "money", "But whenever he gives advice it is always something as startling as "
     "an epigram, and yet as practical as the Bank of England."),
    ("chesterton-ball", "money", "Everybody was ready to believe that the Bank of England might "
     "paint itself pink with white spots."),
    ("edgeworth-parents", "money", "Therefore do me the favour to use the five guinea bank note "
     "which you will find within the ballad."),
    ("edgeworth-parents", "money", "He picked up the five guinea bank note, whilst she read, with "
     "surprise, Susan's Lamentation for her Lamb."),
    ("bryant-stories", "money", "If you ever go to the beautiful city of New Orleans, somebody will "
     "take you down into the old business part of the city, where there are banks and shops and hotels."),
    ("chesterton-brown", "money", "Samuel Harrogate, I arrest you in the name of the law for embezzlement "
     "of the funds of the Hull and Huddersfield Bank."),
]

for fid, sense, sent in bank_sentences:
    print(f"[{sense:5s}] ({fid})  {sent}")

[water] (austen-persuasion)  Anne found a nice seat for her, on a dry sunny bank, under the hedge-row, in which she had no doubt of their still being, in some spot or other.
[water] (bryant-stories)  The minute the fox reached the bank he threw back his head, and gave a snap!
[water] (bryant-stories)  Then he ran swiftly along the river bank, to hunt for crabs; the Camel began to eat sugar-cane.
[water] (burgess-busterbrown)  Little Joe sat down on the bank and prepared to enjoy his breakfast.
[water] (burgess-busterbrown)  He was just climbing up the bank with the fat trout in his mouth.
[water] (carroll-alice)  They were indeed a queer-looking party that assembled on the bank, the birds with draggled feathers, the animals with their fur clinging close to them, and all dripping wet, cross, and uncomfortable.
[water] (chesterton-ball)  He suddenly swung himself up the high bank on one side of the lane.
[water] (edgeworth-parents)  Susan sat down upon the bank in silent sorrow; her litt

**Idea -- Firth's claim, turned into a testable heuristic.** Slide 12's quote said you know a word by the company it keeps. If that is right, the words immediately surrounding "bank" should carry the disambiguating signal, even though the token "bank" itself is identical in every sentence above. Build two small word lists, one for the riverbank sense (river, stream, brook, and so on) and one for the financial sense (dollars, guinea, funds, and so on), then classify each sentence by which list its other words overlap with more. No sentence's label is used by the classifier, only its raw words minus "bank" itself.

In [8]:
WATER_WORDS = {"river", "stream", "brook", "pool", "hedge", "fox", "wet",
               "lane", "trout", "crabs", "sugar-cane", "dripping"}
MONEY_WORDS = {"dollars", "england", "guinea", "note", "business", "shops",
               "hotels", "embezzlement", "funds", "omnibus", "rattling"}

def classify_bank_sentence(sentence):
    words = set(re.findall(r"[a-z'-]+", sentence.lower()))
    water_hits = len(words & WATER_WORDS)
    money_hits = len(words & MONEY_WORDS)
    if water_hits > money_hits:
        return "water"
    if money_hits > water_hits:
        return "money"
    return "undecided"

correct, misses = 0, []
for fid, sense, sent in bank_sentences:
    pred = classify_bank_sentence(sent)
    if pred == sense:
        correct += 1
    else:
        misses.append((fid, sense, pred))
    print(f"true={sense:9s} predicted={pred:9s}  ({fid})")

print(f"\naccuracy: {correct}/{len(bank_sentences)}")
print("missed (all 'undecided', never a wrong-sense guess):", misses)

true=water     predicted=undecided  (austen-persuasion)
true=water     predicted=water      (bryant-stories)
true=water     predicted=water      (bryant-stories)
true=water     predicted=undecided  (burgess-busterbrown)
true=water     predicted=water      (burgess-busterbrown)
true=water     predicted=water      (carroll-alice)
true=water     predicted=water      (chesterton-ball)
true=water     predicted=undecided  (edgeworth-parents)
true=money     predicted=money      (melville-moby_dick)
true=money     predicted=money      (chesterton-thursday)
true=money     predicted=money      (chesterton-thursday)
true=money     predicted=money      (chesterton-ball)
true=money     predicted=money      (edgeworth-parents)
true=money     predicted=money      (edgeworth-parents)
true=money     predicted=money      (bryant-stories)
true=money     predicted=money      (chesterton-brown)

accuracy: 13/16
missed (all 'undecided', never a wrong-sense guess): [('austen-persuasion', 'water', 'undecided'

**Payoff.** Sixteen real sentences, drawn from eight books written across three centuries, split cleanly into two senses just from ordinary reading, confirming slide 8's claim that this is structural, not staged. The neighbour-word heuristic gets 13 of 16 right, and every one of its three misses is an honest "undecided", never a confident wrong-sense guess, because those three sentences ("a dry sunny bank, under the hedge-row", "sat down on the bank and prepared to enjoy his breakfast", "sat down upon the bank in silent sorrow") happen to avoid every word on our two short lists. Firth's intuition survives contact with real data: the words around "bank" really do carry the information needed to tell its senses apart. But look at what did the disambiguating work here: two word lists we typed in by hand, twenty-three words total, tuned to this exact set of sixteen sentences. That does not scale past "bank", and it does not scale past English. Slide 24's unresolved question, how do you learn that kind of context sensitivity automatically, from raw text, with no hand-built lists, is exactly where Session 02 picks up.

## Challenge -- Build a Keyword Q&A System, Then Break It

**Problem.** Parts 1 through 3 tested word counting against Zipf's Law, sentence similarity, and word sense in isolation. A real system has to survive all three problems at once, from a user who has no idea what words the system's documents actually use. Build the simplest thing that could plausibly be called a question-answering system, keyword overlap against a small set of documents, and find out exactly where "simplest thing" stops working.

**Ingredients.** Eight short documents, one fact from this session's own history each: the Turing test, ELIZA, n-gram models, Word2Vec, the attention mechanism, the Transformer, Zipf's Law, and the Winograd Schema. The retrieval function reuses Part 2's exact machinery, a bag-of-words `CountVectorizer` plus cosine similarity, only now comparing one query against eight candidate documents instead of comparing two sentences.

In [9]:
docs = {
    "turing_test": "Alan Turing proposed a test in 1950: if a human judge cannot distinguish "
                   "a machine's conversation from a person's, the machine should be considered "
                   "capable of thought.",
    "eliza": "Joseph Weizenbaum built ELIZA at MIT in 1966, a program that used about two "
             "hundred pattern-matching rules to imitate a psychotherapist.",
    "ngram": "N-gram language models predict the next word by counting how often it follows "
             "the previous few words in a large corpus of text.",
    "word2vec": "Word2Vec, published by Tomas Mikolov in 2013, represents each word as a dense "
                "vector so that words used in similar contexts end up close together in space.",
    "attention_mech": "The attention mechanism, introduced by Bahdanau, Cho, and Bengio around "
                       "2014, lets a neural network learn which parts of a sequence to focus on.",
    "transformer": "The Transformer architecture, introduced in the 2017 paper Attention Is All "
                   "You Need, replaced recurrent networks with attention alone, making training "
                   "far more parallelisable.",
    "zipf": "Zipf's Law describes how a small number of words, like the and of, account for most "
            "of the words used in any text, while most other words occur rarely.",
    "winograd": "The Winograd Schema is a test sentence pair, like the trophy and the suitcase, "
                "where resolving a pronoun requires common sense knowledge about the physical "
                "world, not just grammar.",
}
doc_names = list(docs.keys())
doc_vec = CountVectorizer(stop_words="english")
doc_matrix = doc_vec.fit_transform(docs.values())

def ask(query, show_all=False):
    qv = doc_vec.transform([query])
    sims = cosine_similarity(qv, doc_matrix)[0]
    order = np.argsort(sims)[::-1]
    best = order[0]
    if show_all:
        for i in order:
            print(f"   {sims[i]:.3f}  {doc_names[i]}")
    return doc_names[best], sims[best]

name, score = ask("What did Alan Turing propose in 1950?")
print(f"best match: {name}  (score {score:.3f})")

best match: turing_test  (score 0.420)


**Idea -- try it on questions that echo the documents' own words.** A fair first test: ask about each fact using close to the vocabulary the document itself uses, proper nouns and all. If keyword overlap is going to work anywhere, it should work here.

In [10]:
easy_queries = [
    "What did Alan Turing propose in 1950?",
    "What is Zipf's Law?",
    "What did Word2Vec do in 2013?",
    "What is the Transformer architecture?",
]
for q in easy_queries:
    name, score = ask(q)
    print(f"{score:.3f}  -> {name:16s} | {q}")

0.420  -> turing_test      | What did Alan Turing propose in 1950?
0.316  -> zipf             | What is Zipf's Law?
0.354  -> word2vec         | What did Word2Vec do in 2013?
0.343  -> transformer      | What is the Transformer architecture?


**Payoff.** All four queries retrieve exactly the right document, with clearly positive scores between 0.316 and 0.420. This is the honest case for the system: when a question borrows the document's own proper nouns and technical terms, keyword overlap is a perfectly serviceable retrieval method. Now go break it.

**Idea -- break it deliberately.** Slide 25 asked for exactly this: fifteen minutes spent trying to break your own system, then writing down why it failed. Three angles, straight out of Parts 1-3: a paraphrase that shares no vocabulary with its answer (Part 2's synonym problem), a query built around a word that is ambiguous between an everyday sense and a technical one (Part 3's "bank" problem), and a query about something this eight-document corpus simply does not cover at all.

In [11]:
break_queries = [
    "How did an old chatbot manage to fool people into thinking it was a therapist?",  # paraphrase, no shared vocabulary with 'eliza'
    "How do I get my toddler's attention?",                                             # 'attention' is ambiguous
    "What's the best way to learn to cook pasta?",                                      # not covered by any document at all
]
for q in break_queries:
    print(f"query: {q}")
    ask(q, show_all=True)
    print()

query: How did an old chatbot manage to fool people into thinking it was a therapist?
   0.000  winograd
   0.000  zipf
   0.000  transformer
   0.000  attention_mech
   0.000  word2vec
   0.000  ngram
   0.000  eliza
   0.000  turing_test

query: How do I get my toddler's attention?
   0.485  transformer
   0.267  attention_mech
   0.000  zipf
   0.000  winograd
   0.000  word2vec
   0.000  ngram
   0.000  eliza
   0.000  turing_test

query: What's the best way to learn to cook pasta?
   0.267  attention_mech
   0.000  winograd
   0.000  zipf
   0.000  transformer
   0.000  word2vec
   0.000  ngram
   0.000  eliza
   0.000  turing_test



**Payoff.** All three break attempts land badly, but not in the same way. The ELIZA paraphrase scores 0.000 against *every single* document, an honest sign that nothing matched, yet the retrieval function still returns a document (`winograd`, an arbitrary pick among an eight-way tie) as if it were an answer, because the function was never given a way to say "none of these." The toddler query is worse: it scores 0.485 against `transformer`, the single highest confidence score seen anywhere in this Challenge, higher than any of the four honest queries above, because "attention" appears twice in the Transformer document's text, and cosine similarity rewards that raw overlap with no idea that this query's "attention" and that document's "attention" are different words wearing the same spelling, precisely Part 3's ambiguity problem, now silently breaking a real system instead of a hand-built example. The pasta query is the strangest: nothing in these eight documents is remotely about cooking, yet it still scores 0.267 against `attention_mech`, purely because both texts happen to contain the word "learn". The takeaway that generalises: a system built on exact word overlap cannot distinguish "no good answer exists" from "the right words happened to line up", and it will hand back a confident-looking number in both cases.

### Try It Yourself

Before moving to the reflection, spend a few minutes trying your own break attempts. Type a question below in whichever style you like, paraphrase, jargon, or something unrelated entirely, and see what the system does with it.

In [12]:
query_box = Textarea(placeholder="Type a question for the Q&A system...", layout={"width": "500px"})
run_button = Button(description="Ask")
output_box = Output()

def on_click(_):
    with output_box:
        output_box.clear_output()
        if not query_box.value.strip():
            print("Type a question above, then click Ask.")
            return
        name, score = ask(query_box.value, show_all=True)
        print(f"\nbest match: {name} (score {score:.3f})")

run_button.on_click(on_click)
display(VBox([query_box, run_button, output_box]))

## Reflection

**Problem.** You have now met three of this session's four hard problems as real numbers instead of slide bullets: scale and sparsity in Part 1, the gap between word overlap and meaning in Part 2, and lexical ambiguity in Part 3, then watched all three trip up one small system in the Challenge. The point of this last cell is not a right answer, it is putting the reasoning behind each part into your own words.

In [13]:
# 1. Part 1 showed the corpus's vocabulary was still growing, about one new
#    word every 48 tokens, even after 2 million tokens of real prose. If you
#    fed this same counting exercise ten times more text, would you expect
#    that "1 new word every N tokens" rate to reach infinity (never happen
#    again) or just keep climbing slowly? What does your answer imply for
#    a model that has to handle text it was never trained on?
answer_1 = """
"""

# 2. In Part 2, the two "different meaning" sentence pairs that scored a
#    perfect 1.0 (dog/cat, doctor/nurse) were both built by swapping the
#    subject and object of the same words. Name the specific piece of
#    information a bag-of-words vector throws away that would be needed to
#    tell these two sentences apart.
answer_2 = """
"""

# 3. Part 3's neighbour-word classifier reached 13/16 by hand-building two
#    word lists tuned to sixteen specific sentences. Firth's quote, "you
#    shall know a word by the company it keeps," was validated by this
#    result. Explain, in your own words, why the validation and the
#    "does not scale" problem are not in tension: both things are true at
#    the same time.
answer_3 = """
"""

# 4. The Challenge's toddler-attention query scored HIGHER (0.485) than any
#    of the four honest, on-topic queries in the easy-queries test. Why is
#    a wrong answer with high confidence more dangerous in a deployed
#    system than a wrong answer that at least scores low?
answer_4 = """
"""

# 5. Across Parts 1-3 and the Challenge, you used exactly one representation
#    idea: count which words are present (as raw text, as a bag-of-words
#    vector, as a hand-built keyword list). Name one specific piece of
#    information that idea never has access to, no matter how it is
#    dressed up, and that you expect Session 02's word vectors to finally
#    capture.
answer_5 = """
"""

---
*Why Language Is Hard · Bluevision-ai Academy*